# Step 1 — Mutational Potential V_mut (Calculations V1–V4)

**Purpose:** Characterise the intrinsic mutability landscape of each germline V gene:
- **V1** — S5F mutability model: per-position 5-mer mutability scores derived from synonymous mutations in memory sequences
- **V2** — AID hotspot enrichment: CDR vs FWR hotspot density ratio per germline (R_AID)
- **V3** — CDRH3 length vs V-region SHM load
- **V4** — Mutation accumulation by isotype

**Input:** `processed/aligned_master.parquet`  
**Outputs:** `processed/mutability_profiles.parquet`, `results/tables/`, `results/figures/`  
**Design reference:** DESIGN.md §Step 1 (V1–V4)

In [ ]:
import polars as pl
import numpy as np
from pathlib import Path
import re
from collections import defaultdict
import time

In [ ]:
DATA_DIR   = Path("/home/jovyan/shared/Benjamin/LineageAtlas/pairplex_paper/")
PROC_DIR   = DATA_DIR / "processed"
RESULTS    = DATA_DIR / "results"
FIGURES    = RESULTS / "figures"
TABLES     = RESULTS / "tables"

for d in [RESULTS, FIGURES, TABLES]:
    d.mkdir(parents=True, exist_ok=True)

MASTER_FILE       = PROC_DIR / "aligned_master.parquet"
MUTABILITY_FILE   = PROC_DIR / "mutability_profiles.parquet"

print("Paths OK")

## 1.0  Load master table

In [ ]:
master = pl.read_parquet(MASTER_FILE)
print(f"Master table: {master.shape}")
print(f"  naive_strict : {master.filter(pl.col('naive_strict')).height:,}")
print(f"  memory       : {master.filter(~pl.col('naive_bio') & ~pl.col('naive_comp')).height:,}")

# Memory-only subset used for all mutability derivations
memory = master.filter(~pl.col('naive_bio') & ~pl.col('naive_comp'))
print(f"\nMemory sequences: {memory.height:,}")

## 1.1  Aho region maps

Reproduced from Step 0 so this notebook runs standalone.

In [ ]:
def aho_to_nt_cols(chain: str, aho_start: int, aho_end: int) -> list[str]:
    return [f"{chain}{i}" for i in range((aho_start - 1) * 3 + 1, aho_end * 3 + 1)]

H_REGIONS = {
    'FR1' : aho_to_nt_cols('H',   1,  25),
    'CDR1': aho_to_nt_cols('H',  26,  38),
    'FR2' : aho_to_nt_cols('H',  39,  49),
    'CDR2': aho_to_nt_cols('H',  50,  64),
    'FR3' : aho_to_nt_cols('H',  65, 108),
    'CDR3': aho_to_nt_cols('H', 109, 137),
    'FR4' : aho_to_nt_cols('H', 138, 149),
}
L_REGIONS = {
    'FR1' : aho_to_nt_cols('L',   1,  25),
    'CDR1': aho_to_nt_cols('L',  26,  38),
    'FR2' : aho_to_nt_cols('L',  39,  49),
    'CDR2': aho_to_nt_cols('L',  50,  66),
    'FR3' : aho_to_nt_cols('L',  67, 108),
    'CDR3': aho_to_nt_cols('L', 109, 138),
    'FR4' : aho_to_nt_cols('L', 139, 148),
}

# Flat CDR / FWR column lists (excluding CDR3 for V-gene-only analyses)
H_CDR_V = H_REGIONS['CDR1'] + H_REGIONS['CDR2']   # CDR1+CDR2 only (no CDR3 in V gene)
H_FWR_V = H_REGIONS['FR1']  + H_REGIONS['FR2'] + H_REGIONS['FR3']
H_ALL_V  = H_FWR_V + H_CDR_V  # full V gene (FR1-FR3 + CDR1-CDR2)

L_CDR_V = L_REGIONS['CDR1'] + L_REGIONS['CDR2']
L_FWR_V = L_REGIONS['FR1']  + L_REGIONS['FR2'] + L_REGIONS['FR3']
L_ALL_V  = L_FWR_V + L_CDR_V

print(f"H V-gene region (excl. CDR3): {len(H_ALL_V)//3} codons  "
      f"(CDR={len(H_CDR_V)//3}, FWR={len(H_FWR_V)//3})")
print(f"L V-gene region (excl. CDR3): {len(L_ALL_V)//3} codons  "
      f"(CDR={len(L_CDR_V)//3}, FWR={len(L_FWR_V)//3})")

## V1 — S5F Mutability Model

### Strategy

The S5F model assigns per-nucleotide mutability based on the 5-mer context centred on each
position. Rather than importing the shazam R package, we derive the model empirically from
**synonymous mutations in memory sequences** (synonymous = nucleotide changed but amino acid
preserved). This avoids selection bias: synonymous mutations are nearly neutral, so their
rate reflects intrinsic AID/Polη targeting, not fitness effects.

**Steps:**
1. For each germline sequence, extract the 5-mer context at every V-gene position
2. Count observed synonymous mutations at each 5-mer context across all memory sequences
3. Count total opportunities (positions seen × number of sequences) per 5-mer
4. Mutability score = observed synonymous mutations / total opportunities
5. Normalise relative to the genome-wide mean

### V1.1  Build germline V-gene nucleotide sequences

We reconstruct one consensus germline sequence per IGHV gene from the `germline:0` column in
the master table (same germline used for alignment). Each germline is the canonical
nucleotide string stored by iggnition.

We use the raw germline strings rather than the Aho-aligned wide table so that:
1. We avoid loading the heavy alignment files here
2. Consecutive nucleotide context windows are guaranteed contiguous

In [ ]:
# One germline sequence per IGHV gene (use the full V-gene germline string)
# Use the V-region germline (v_germline:0) which covers FR1–FR3+CDR1–CDR2
germline_seqs = (
    master
    .select(['v_gene:0', 'v_germline:0'])
    .drop_nulls()
    .group_by('v_gene:0')
    .agg(pl.col('v_germline:0').first())  # one representative per gene
    .rename({'v_gene:0': 'v_gene', 'v_germline:0': 'germline_nt'})
    .sort('v_gene')
)

print(f"Unique IGHV genes: {germline_seqs.height}")
print(germline_seqs.head(5))

### V1.2  Extract synonymous mutations from memory sequences

We need per-position synonymous mutation counts across all memory sequences.
The master table has `n_S_CDR_H` / `n_S_FWR_H` summary counts, but for the S5F model
we need position-level resolution. We load the Aho-aligned sequence parquets to extract
per-position mutation identity for each memory sequence.

For the 5-mer model we need:
- The germline nucleotide at position i and its ±2 flanking positions (5-mer context)
- Whether the observed mutation at i was synonymous

In [ ]:
# Load cached iggnition alignment tables
MUTATED_FILE  = PROC_DIR / "iggnition_sequences.parquet"
GERMLINE_FILE = PROC_DIR / "iggnition_germlines.parquet"

print("Loading sequence alignment...")
mutated  = pl.read_parquet(MUTATED_FILE)
germline = pl.read_parquet(GERMLINE_FILE)

KEY = "seq_name"

# Deduplicate (same as Step 0)
mutated_dedup  = mutated.unique(subset=[KEY], keep='first')
germline_dedup = germline.unique(subset=[KEY], keep='first')

# Filter to memory sequences only
memory_names = set(memory[KEY].to_list())
print(f"Memory seq_names: {len(memory_names):,}")

mut_mem  = mutated_dedup.filter(pl.col(KEY).is_in(memory_names))
germ_mem = germline_dedup.filter(pl.col(KEY).is_in(memory_names))

print(f"Memory sequences in alignment: seq={mut_mem.height:,}, germ={germ_mem.height:,}")

In [ ]:
# Build aligned joint for memory sequences only
pos_cols = [c for c in mutated.columns if c != KEY]

aligned_mem = (
    mut_mem
    .join(germ_mem, on=KEY, how='inner', suffix='_germ')
    .sort(KEY)
)
print(f"Memory aligned joint: {aligned_mem.height:,} rows")

# Extract numpy arrays for H chain V-region (FR1–FR3 + CDR1–CDR2, excl. CDR3)
H_V_COLS_SEQ  = [c for c in H_ALL_V  if c in set(aligned_mem.columns)]
H_V_COLS_GERM = [f"{c}_germ" for c in H_V_COLS_SEQ if f"{c}_germ" in set(aligned_mem.columns)]
H_V_COLS_SEQ  = [c for c in H_V_COLS_SEQ if f"{c}_germ" in set(aligned_mem.columns)]

print(f"H V-region nt columns: {len(H_V_COLS_SEQ)}")

seq_HV  = aligned_mem.select(H_V_COLS_SEQ).fill_null(0).to_numpy().astype(np.int32)
germ_HV = aligned_mem.select(H_V_COLS_GERM).fill_null(0).to_numpy().astype(np.int32)
print(f"Arrays: seq={seq_HV.shape}, germ={germ_HV.shape}")

In [ ]:
# Genetic code (same as Step 0)
GENETIC_CODE = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L',
    'CTT':'L','CTC':'L','CTA':'L','CTG':'L',
    'ATT':'I','ATC':'I','ATA':'I','ATG':'M',
    'GTT':'V','GTC':'V','GTA':'V','GTG':'V',
    'TCT':'S','TCC':'S','TCA':'S','TCG':'S',
    'CCT':'P','CCC':'P','CCA':'P','CCG':'P',
    'ACT':'T','ACC':'T','ACA':'T','ACG':'T',
    'GCT':'A','GCC':'A','GCA':'A','GCG':'A',
    'TAT':'Y','TAC':'Y','TAA':'*','TAG':'*',
    'CAT':'H','CAC':'H','CAA':'Q','CAG':'Q',
    'AAT':'N','AAC':'N','AAA':'K','AAG':'K',
    'GAT':'D','GAC':'D','GAA':'E','GAG':'E',
    'TGT':'C','TGC':'C','TGA':'*','TGG':'W',
    'CGT':'R','CGC':'R','CGA':'R','CGG':'R',
    'AGT':'S','AGC':'S','AGA':'R','AGG':'R',
    'GGT':'G','GGC':'G','GGA':'G','GGG':'G',
}
AA_LOOKUP = {0: 0}
for codon, aa in GENETIC_CODE.items():
    k = ord(codon[0]) * 65536 + ord(codon[1]) * 256 + ord(codon[2])
    AA_LOOKUP[k] = ord(aa)
_aa_vec = np.vectorize(lambda k: AA_LOOKUP.get(int(k), 0), otypes=[np.int32])

def translate_codon_array(nt: np.ndarray) -> np.ndarray:
    """(N, 3) int32 → (N,) AA ordinals; 0 = null/invalid."""
    keys = nt[:, 0].astype(np.int32) * 65536 + nt[:, 1].astype(np.int32) * 256 + nt[:, 2].astype(np.int32)
    return _aa_vec(keys)

# Integer → nucleotide character
INT_TO_NT = {65: 'A', 67: 'C', 71: 'G', 84: 'T'}

print("Genetic code loaded.")

In [ ]:
# Per-position synonymous mutation counts and opportunities in the H V-region
# For each position p (0-indexed within H_V_COLS_SEQ):
#   - opportunity = sequences where germ and seq both have valid (non-null, non-gap) nt at this position
#   - syn_mut = sequences with a synonymous mutation at the CODON containing position p
#
# We work at codon granularity: for codon c (containing positions 3c, 3c+1, 3c+2),
# label each of the 3 positions as synonymous if the codon has a nt change but same AA.

N = seq_HV.shape[0]
GAP = 45

print(f"Processing {len(H_V_COLS_SEQ)//3} codons across {N:,} memory sequences...")
t0 = time.time()

# Output: per nt-position lists
pos_opportunity = np.zeros(len(H_V_COLS_SEQ), dtype=np.int64)
pos_syn_mut     = np.zeros(len(H_V_COLS_SEQ), dtype=np.int64)
pos_germ_nt     = np.full(len(H_V_COLS_SEQ), '', dtype=object)  # most common germline nt

for codon_i in range(len(H_V_COLS_SEQ) // 3):
    p0, p1, p2 = codon_i * 3, codon_i * 3 + 1, codon_i * 3 + 2

    g = germ_HV[:, p0:p2+1]   # (N, 3) germline nt
    s = seq_HV[:,  p0:p2+1]   # (N, 3) sequence nt

    # Valid positions: both germ and seq non-null and non-gap at all 3 positions
    valid = (np.all(g > 0, axis=1) & np.all(g != GAP, axis=1) &
             np.all(s > 0, axis=1) & np.all(s != GAP, axis=1))

    has_mut = valid & np.any(g != s, axis=1)
    g_aa = translate_codon_array(g)
    s_aa = translate_codon_array(s)
    is_syn = has_mut & (g_aa == s_aa) & (g_aa > 0) & (s_aa > 0)

    # Assign opportunity and syn_mut to each nt position in the codon
    for p in [p0, p1, p2]:
        pos_opportunity[p] = valid.sum()
        pos_syn_mut[p]     = is_syn.sum()

        # Most common germline nucleotide at this position (from valid sequences)
        gnt = germ_HV[valid, p]
        if len(gnt) > 0:
            vals, counts = np.unique(gnt, return_counts=True)
            best = vals[np.argmax(counts)]
            pos_germ_nt[p] = INT_TO_NT.get(int(best), '?')

print(f"Done in {time.time()-t0:.1f}s")
print(f"Total synonymous mutations counted: {pos_syn_mut.sum():,}")
print(f"Total opportunities: {pos_opportunity.sum():,}")

In [ ]:
# For each position, compute the 5-mer context from the germline sequence.
# We use the consensus germline nt already computed above (pos_germ_nt).
# Positions within 2 of the ends get a shorter context (or can be skipped).

def fivemer(germ_nts: list[str], pos: int) -> str:
    """Return 5-mer context centred on pos, using 'N' for out-of-bounds."""
    context = []
    for offset in [-2, -1, 0, 1, 2]:
        i = pos + offset
        if 0 <= i < len(germ_nts):
            context.append(germ_nts[i] if germ_nts[i] else 'N')
        else:
            context.append('N')
    return ''.join(context)

germ_nt_list = pos_germ_nt.tolist()
fivemerctx = [fivemer(germ_nt_list, p) for p in range(len(H_V_COLS_SEQ))]

# Assign region label per position
region_labels = []
region_order = [('FR1', H_REGIONS['FR1']), ('CDR1', H_REGIONS['CDR1']),
                ('FR2', H_REGIONS['FR2']), ('CDR2', H_REGIONS['CDR2']),
                ('FR3', H_REGIONS['FR3'])]
col_to_region = {}
for reg_name, reg_cols in region_order:
    for c in reg_cols:
        col_to_region[c] = reg_name
region_labels = [col_to_region.get(c, 'unknown') for c in H_V_COLS_SEQ]

print(f"5-mer contexts computed for {len(fivemerctx)} positions")
print(f"Example (position 0): {fivemerctx[0]!r}")
print(f"Example (position 100): {fivemerctx[100]!r}")

In [ ]:
# Aggregate per-5-mer mutability across all H V-gene positions
fivemer_syn   = defaultdict(int)
fivemer_opps  = defaultdict(int)

for p in range(len(H_V_COLS_SEQ)):
    ctx = fivemerctx[p]
    if 'N' not in ctx:  # skip edge positions with incomplete context
        fivemer_syn[ctx]  += pos_syn_mut[p]
        fivemer_opps[ctx] += pos_opportunity[p]

# Compute mutability score per 5-mer
all_fivemers = sorted(set(fivemer_opps.keys()))
mut_score = {
    ctx: (fivemer_syn[ctx] / fivemer_opps[ctx]) if fivemer_opps[ctx] > 0 else 0.0
    for ctx in all_fivemers
}

# Normalise to mean = 1
scores = np.array(list(mut_score.values()))
mean_score = scores[scores > 0].mean()
for ctx in mut_score:
    if mean_score > 0:
        mut_score[ctx] /= mean_score

print(f"Unique 5-mer contexts observed: {len(all_fivemers)}")
print(f"Mean mutability (post-normalisation): {np.mean(list(mut_score.values())):.4f}")
print(f"Max mutability: {max(mut_score.values()):.4f}")

In [ ]:
# Classify each position by hotspot type based on the germline 5-mer context.
# AID primary targets: WRC (W=A/T, R=A/G) and its complement RGYW (R=A/G, Y=C/T, W=A/T)
# Polη targets: WA and TW
# SYC coldspot: S=C/G, Y=C/T

def classify_hotspot(ctx: str) -> str:
    """Classify 5-mer by AID/Polη hotspot type. ctx is 5-mer, central nt at index 2."""
    if len(ctx) != 5 or 'N' in ctx:
        return 'edge'
    c = ctx  # positions: c[0]=i-2, c[1]=i-1, c[2]=i, c[3]=i+1, c[4]=i+2
    W  = set('AT')
    R  = set('AG')
    Y  = set('CT')
    S  = set('CG')
    # WRC: positions i-1=W, i=R, i+1=C → central = R
    if c[1] in W and c[2] in R and c[3] == 'C':
        return 'WRC'
    # RGYW: i-1=R, i=G, i+1=Y, i+2=W → but central must be G
    if c[1] in R and c[2] == 'G' and c[3] in Y and c[4] in W:
        return 'RGYW'
    # WA: i-1=W, i=A
    if c[1] in W and c[2] == 'A':
        return 'WA'
    # TW: i=T, i+1=W
    if c[2] == 'T' and c[3] in W:
        return 'TW'
    # SYC coldspot: i-1=S, i=Y, i+1=C
    if c[1] in S and c[2] in Y and c[3] == 'C':
        return 'SYC_cold'
    return 'neutral'

hotspot_labels = [classify_hotspot(ctx) for ctx in fivemerctx]

from collections import Counter
hs_counts = Counter(hotspot_labels)
print("Hotspot type distribution (H V-region positions):")
for hs, n in sorted(hs_counts.items(), key=lambda x: -x[1]):
    print(f"  {hs:12s}: {n:5d} ({100*n/len(hotspot_labels):.1f}%)")

In [ ]:
# Build per-position mutability profile for the H V-gene
# (positions = Aho nt column names from H_V_COLS_SEQ)

# Aho amino-acid position for each nt position
def aho_aa_pos(col: str) -> int:
    """Return Aho AA position for a nt column name like 'H76' → 26."""
    nt_idx = int(col[1:])  # e.g. 76
    return (nt_idx - 1) // 3 + 1

raw_scores = np.array([pos_syn_mut[p] / pos_opportunity[p]
                       if pos_opportunity[p] > 0 else 0.0
                       for p in range(len(H_V_COLS_SEQ))])
global_mean = raw_scores[raw_scores > 0].mean()
norm_scores = np.where(raw_scores > 0, raw_scores / global_mean, 0.0)

profile_H = pl.DataFrame({
    'chain':          ['H'] * len(H_V_COLS_SEQ),
    'nt_col':         H_V_COLS_SEQ,
    'aho_nt_pos':     [int(c[1:]) for c in H_V_COLS_SEQ],
    'aho_aa_pos':     [aho_aa_pos(c) for c in H_V_COLS_SEQ],
    'region':         region_labels,
    'germ_nt':        pos_germ_nt.tolist(),
    'fivemer':        fivemerctx,
    'hotspot_type':   hotspot_labels,
    'n_syn_mut':      pos_syn_mut.tolist(),
    'n_opportunity':  pos_opportunity.tolist(),
    'mutability_raw': raw_scores.tolist(),
    'mutability':     norm_scores.tolist(),
})

print(f"H mutability profile: {profile_H.shape}")
print(profile_H.head(6))

In [ ]:
# Save profile
profile_H.write_parquet(MUTABILITY_FILE)
profile_H.write_csv(TABLES / "mutability_profile_H.csv")
print(f"Saved → {MUTABILITY_FILE}")
print(f"Saved → {TABLES / 'mutability_profile_H.csv'}")

## V1 — Plot: Mutability profile across H V-region positions

One cell = one plot. Data saved as CSV alongside the figure.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Data ────────────────────────────────────────────────────────────────────
profile = pl.read_parquet(MUTABILITY_FILE)  # standalone-safe: loads from file

plot_data = profile.select(['aho_nt_pos', 'region', 'hotspot_type', 'mutability'])
plot_data.write_csv(FIGURES / "fig_v1_mutability_profile.csv")

# ── Figure ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))

x     = plot_data['aho_nt_pos'].to_numpy()
y     = plot_data['mutability'].to_numpy()
regs  = plot_data['region'].to_list()
hs    = plot_data['hotspot_type'].to_list()

# Region shading
REGION_COLORS = {'CDR1': '#FFCCCC', 'CDR2': '#FFCCCC', 'FR1': '#E8F4FD',
                 'FR2': '#E8F4FD', 'FR3': '#E8F4FD'}
current_reg, reg_start = regs[0], x[0]
for i in range(1, len(regs)):
    if regs[i] != current_reg or i == len(regs) - 1:
        col = REGION_COLORS.get(current_reg, '#FFFFFF')
        ax.axvspan(reg_start, x[i-1], color=col, alpha=0.4, lw=0)
        current_reg = regs[i]
        reg_start = x[i]

# Line
ax.plot(x, y, color='#333333', lw=0.8, alpha=0.7)

# Hotspot highlights
HS_COLORS = {'WRC': '#E53935', 'RGYW': '#FB8C00', 'WA': '#8E24AA',
             'TW': '#5E35B1', 'SYC_cold': '#1E88E5', 'neutral': None}
for xi, yi, h in zip(x, y, hs):
    c = HS_COLORS.get(h)
    if c:
        ax.scatter(xi, yi, color=c, s=10, zorder=3, linewidths=0)

ax.set_xlabel('Aho nt position')
ax.set_ylabel('Relative mutability (synonymous)')
ax.set_title('H V-region per-position mutability (S5F-derived from memory synonymous mutations)')
ax.axhline(1.0, color='gray', linestyle='--', lw=0.8, label='Neutral (mean=1)')

patches = [mpatches.Patch(color=c, label=l)
           for l, c in HS_COLORS.items() if c is not None]
patches.append(mpatches.Patch(color='#FFCCCC', alpha=0.4, label='CDR'))
patches.append(mpatches.Patch(color='#E8F4FD', alpha=0.4, label='FWR'))
ax.legend(handles=patches, fontsize=7, ncol=4, loc='upper right')

plt.tight_layout()
plt.savefig(FIGURES / "fig_v1_mutability_profile.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

## V2 — AID Hotspot Enrichment per Germline (R_AID)

For each IGHV gene: ratio of WRC/RGYW density in CDRs vs FWRs.
High R_AID = CDRs are enriched for AID hotspots → germline is "pre-wired" for CDR targeting.

In [ ]:
# Per-germline hotspot analysis
# For each germline sequence: extract WRC/RGYW density in CDR1+CDR2 vs FR1+FR2+FR3

AID_HOTSPOTS = {'WRC', 'RGYW'}

# Map each H_V position to CDR vs FWR
CDR_SET = set(H_CDR_V)
FWR_SET = set(H_FWR_V)

# Build per-germline hotspot counts directly from the profile table
# We leverage the consensus germline nt in profile_H as representative for each germline
# For per-germline analysis, we need per-gene germline sequences → use iggnition aligned germlines
# grouped by v_gene

germ_cols = [c for c in H_V_COLS_SEQ if c in set(germline_dedup.columns)]

v_gene_col = 'v_gene:0'

# Join v_gene to germline alignment via seq_name
germ_vgene = (
    master.select([KEY, v_gene_col])
    .join(germline_dedup.select([KEY] + germ_cols), on=KEY, how='inner')
)

# Compute consensus germline per V gene (mode per position)
print("Computing per-germline hotspot enrichment...")
t0 = time.time()

records = []
for v_gene_name, grp in germ_vgene.group_by(v_gene_col):
    if grp.height < 10:
        continue  # skip rare genes
    v_gene_name = v_gene_name[0] if isinstance(v_gene_name, tuple) else v_gene_name
    arr = grp.select(germ_cols).to_numpy().astype(np.int32)

    # Consensus: most frequent non-null non-gap nt per position
    cons = []
    for p in range(arr.shape[1]):
        col = arr[:, p]
        valid_nt = col[(col > 0) & (col != GAP)]
        if len(valid_nt) == 0:
            cons.append('')
        else:
            vals, cnts = np.unique(valid_nt, return_counts=True)
            cons.append(INT_TO_NT.get(int(vals[np.argmax(cnts)]), ''))

    # 5-mer contexts for this germline
    n_wrc_cdr, n_wrc_fwr = 0, 0
    n_cdr, n_fwr = 0, 0
    for p, col in enumerate(germ_cols):
        ctx = fivemer(cons, p)
        hs  = classify_hotspot(ctx)
        in_cdr = col in CDR_SET
        in_fwr = col in FWR_SET
        if in_cdr:
            n_cdr += 1
            if hs in AID_HOTSPOTS:
                n_wrc_cdr += 1
        elif in_fwr:
            n_fwr += 1
            if hs in AID_HOTSPOTS:
                n_wrc_fwr += 1

    density_cdr = n_wrc_cdr / n_cdr if n_cdr > 0 else 0.0
    density_fwr = n_wrc_fwr / n_fwr if n_fwr > 0 else 0.0
    r_aid = density_cdr / density_fwr if density_fwr > 0 else np.nan

    records.append({
        'v_gene': v_gene_name,
        'n_sequences': grp.height,
        'n_wrc_cdr': n_wrc_cdr,
        'n_wrc_fwr': n_wrc_fwr,
        'n_cdr_positions': n_cdr,
        'n_fwr_positions': n_fwr,
        'density_cdr': density_cdr,
        'density_fwr': density_fwr,
        'R_AID': r_aid,
    })

evolvability = pl.DataFrame(records).sort('R_AID', descending=True)
print(f"Done in {time.time()-t0:.1f}s — {evolvability.height} germlines")
print(evolvability.head(10))

In [ ]:
evolvability.write_csv(TABLES / "evolvability_index.csv")
print(f"Saved → {TABLES / 'evolvability_index.csv'}")

In [ ]:
import matplotlib.pyplot as plt

# ── Data ────────────────────────────────────────────────────────────────────
ev = pl.read_csv(TABLES / "evolvability_index.csv")  # standalone-safe

plot_data = (
    ev.filter(pl.col('n_sequences') >= 100)  # require ≥100 sequences for reliable estimate
    .sort('R_AID', descending=True)
    .head(40)
)
plot_data.write_csv(FIGURES / "fig_v2_raid_top40.csv")

# ── Figure ──────────────────────────────────────────────────────────────────
highlight = {'IGHV3-23', 'IGHV1-2', 'IGHV1-69', 'IGHV4-34'}

genes  = plot_data['v_gene'].to_list()
raid   = plot_data['R_AID'].to_numpy()
colors = ['#E53935' if g in highlight else '#1E88E5' for g in genes]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.barh(range(len(genes)), raid, color=colors, edgecolor='none')
ax.set_yticks(range(len(genes)))
ax.set_yticklabels(genes, fontsize=7)
ax.axvline(1.0, color='gray', linestyle='--', lw=1, label='Neutral (R_AID=1)')
ax.set_xlabel('R_AID  (CDR hotspot density / FWR hotspot density)')
ax.set_title('AID hotspot enrichment in CDRs vs FWRs per IGHV germline (top 40)')
ax.legend()

import matplotlib.patches as mpatches
patches = [mpatches.Patch(color='#E53935', label='Known evolvable / notable'),
           mpatches.Patch(color='#1E88E5', label='Other')]
ax.legend(handles=patches)

plt.tight_layout()
plt.savefig(FIGURES / "fig_v2_raid_top40.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

## V3 — CDRH3 Length vs V-region SHM Load

Does longer CDRH3 correlate with higher V-region mutation burden?
Analysis on memory sequences only; V-region SHM = n_mut_H (includes FR1-FR3 + CDR1-CDR2 mutations).

In [ ]:
from scipy import stats

# Bin memory sequences by CDRH3 length (2-aa bins)
bins = list(range(6, 30, 2)) + [30]
bin_labels = [f"{bins[i]}-{bins[i+1]-1}" for i in range(len(bins)-1)]

mem_data = (
    memory
    .select(['cdrh3_length', 'n_mut_H', 'v_gene:0'])
    .filter(pl.col('cdrh3_length') > 0)
)

# Assign bin
mem_np = mem_data.to_numpy()
cdrh3_lens = mem_data['cdrh3_length'].to_numpy().astype(float)
n_mut_H    = mem_data['n_mut_H'].to_numpy().astype(float)

bin_stats = []
for i in range(len(bins)-1):
    mask = (cdrh3_lens >= bins[i]) & (cdrh3_lens < bins[i+1])
    vals = n_mut_H[mask]
    if len(vals) > 10:
        bin_stats.append({
            'bin_label': bin_labels[i],
            'cdrh3_length_mid': (bins[i] + bins[i+1]) / 2,
            'n': int(mask.sum()),
            'mean_n_mut_H': float(vals.mean()),
            'std_n_mut_H':  float(vals.std()),
            'median_n_mut_H': float(np.median(vals)),
        })

bin_df = pl.DataFrame(bin_stats)

# Linear regression: n_mut_H ~ cdrh3_length
slope, intercept, r_value, p_value, std_err = stats.linregress(cdrh3_lens, n_mut_H)
print(f"Linear regression: slope={slope:.4f}, R²={r_value**2:.4f}, p={p_value:.2e}")
print(bin_df)

In [ ]:
bin_df.write_csv(TABLES / "cdrh3_length_vs_shm.csv")
print(f"Saved → {TABLES / 'cdrh3_length_vs_shm.csv'}")

In [ ]:
import matplotlib.pyplot as plt

# ── Data ────────────────────────────────────────────────────────────────────
df = pl.read_csv(TABLES / "cdrh3_length_vs_shm.csv")  # standalone-safe

plot_data = df.select(['cdrh3_length_mid', 'mean_n_mut_H', 'std_n_mut_H', 'n'])
plot_data.write_csv(FIGURES / "fig_v3_cdrh3_length_vs_shm.csv")

x      = df['cdrh3_length_mid'].to_numpy()
y_mean = df['mean_n_mut_H'].to_numpy()
y_std  = df['std_n_mut_H'].to_numpy()

# ── Figure ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(x, y_mean, yerr=y_std, fmt='o-', color='#1E88E5',
            capsize=4, linewidth=1.5, markersize=5, label='Mean ± SD')

# Regression line
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, slope * x_line + intercept, '--', color='#E53935',
        lw=1.5, label=f'Linear fit  R²={r_value**2:.3f}  p={p_value:.2e}')

ax.set_xlabel('CDRH3 length (aa)')
ax.set_ylabel('Mean VH codon mutations (V-region)')
ax.set_title('CDRH3 length vs VH mutation load (memory sequences)')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "fig_v3_cdrh3_length_vs_shm.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

## V4 — Mutation Accumulation by Isotype

Isotype is used as a proxy for maturation depth:
`IgM < IgG < IgA ≤ IgE` (approximate GC transit time order).

We compute mean and distribution of VH codon mutations per isotype, then fit a Poisson model
to estimate relative maturation time λ(isotype).

In [ ]:
from scipy.optimize import minimize_scalar
from scipy.special import gammaln

# Isotype from c_gene:0 (heavy chain constant region)
# Collapse IgG1/2/3/4 and IgA1/2 into IgG and IgA
def coarse_isotype(c_gene: str) -> str:
    if c_gene is None:
        return 'Unknown'
    c = str(c_gene).upper()
    if c.startswith('IGHM'):  return 'IgM'
    if c.startswith('IGHG'):  return 'IgG'
    if c.startswith('IGHA'):  return 'IgA'
    if c.startswith('IGHE'):  return 'IgE'
    if c.startswith('IGHD'):  return 'IgD'
    return 'Other'

mem_iso = (
    memory
    .with_columns(
        pl.col('c_gene:0').map_elements(coarse_isotype, return_dtype=pl.Utf8).alias('isotype')
    )
    .select(['isotype', 'n_mut_H'])
)

isotype_stats = (
    mem_iso
    .group_by('isotype')
    .agg([
        pl.len().alias('n'),
        pl.col('n_mut_H').mean().alias('mean_n_mut_H'),
        pl.col('n_mut_H').std().alias('std_n_mut_H'),
        pl.col('n_mut_H').median().alias('median_n_mut_H'),
    ])
    .sort('mean_n_mut_H')
)
print(isotype_stats)

# Poisson MLE: for each isotype, lambda_hat = sample mean
# (Poisson MLE for lambda given observed counts x_1..x_n is just the sample mean)
poisson_lambda = {
    row['isotype']: row['mean_n_mut_H']
    for row in isotype_stats.to_dicts()
}
print("\nPoisson lambda (=mean) per isotype:")
for iso, lam in sorted(poisson_lambda.items(), key=lambda x: x[1]):
    print(f"  {iso:8s}: {lam:.2f}")

In [ ]:
isotype_stats.write_csv(TABLES / "isotype_mutation_depth.csv")
print(f"Saved → {TABLES / 'isotype_mutation_depth.csv'}")

In [ ]:
import matplotlib.pyplot as plt

# ── Data ────────────────────────────────────────────────────────────────────
df = pl.read_csv(TABLES / "isotype_mutation_depth.csv")  # standalone-safe

plot_data = df.select(['isotype', 'n', 'mean_n_mut_H', 'std_n_mut_H', 'median_n_mut_H'])
plot_data.write_csv(FIGURES / "fig_v4_isotype_mutation_depth.csv")

# ── Figure: box-style bar with error bars ───────────────────────────────────
ISO_ORDER = ['IgM', 'IgD', 'IgG', 'IgA', 'IgE', 'Other', 'Unknown']
df_dict = {r['isotype']: r for r in df.to_dicts()}
orderedISO = [iso for iso in ISO_ORDER if iso in df_dict]

means  = [df_dict[iso]['mean_n_mut_H']  for iso in orderedISO]
stds   = [df_dict[iso]['std_n_mut_H']   for iso in orderedISO]
counts = [df_dict[iso]['n']             for iso in orderedISO]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(orderedISO, means, yerr=stds, capsize=5,
              color=['#90CAF9','#CE93D8','#A5D6A7','#FFCC80','#EF9A9A','#BDBDBD','#EEEEEE'],
              edgecolor='none', error_kw={'elinewidth': 1.2})
for i, (iso, n, m) in enumerate(zip(orderedISO, counts, means)):
    ax.text(i, m + stds[i] + 0.3, f'n={n:,}\nλ={m:.1f}',
            ha='center', va='bottom', fontsize=7)

ax.set_ylabel('Mean VH codon mutations')
ax.set_xlabel('Isotype')
ax.set_title('VH mutation load by isotype (memory sequences, Poisson proxy for maturation depth)')
plt.tight_layout()
plt.savefig(FIGURES / "fig_v4_isotype_mutation_depth.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

In [ ]:
import matplotlib.pyplot as plt

# ── Data ────────────────────────────────────────────────────────────────────
# Distribution (full counts) — saved per-isotype for reproducibility
ISO_ORDER_CORE = ['IgM', 'IgG', 'IgA', 'IgE']
iso_dist_records = []
for iso in ISO_ORDER_CORE:
    vals = (
        memory
        .with_columns(pl.col('c_gene:0').map_elements(coarse_isotype, return_dtype=pl.Utf8).alias('isotype'))
        .filter(pl.col('isotype') == iso)
        ['n_mut_H']
        .to_numpy()
    )
    for v in vals:
        iso_dist_records.append({'isotype': iso, 'n_mut_H': int(v)})

iso_dist_df = pl.DataFrame(iso_dist_records)
iso_dist_df.write_csv(FIGURES / "fig_v4_isotype_mutation_distributions.csv")

# ── Figure ──────────────────────────────────────────────────────────────────
ISO_COLORS = {'IgM': '#90CAF9', 'IgG': '#A5D6A7', 'IgA': '#FFCC80', 'IgE': '#EF9A9A'}
fig, ax = plt.subplots(figsize=(10, 5))
for iso in ISO_ORDER_CORE:
    vals = iso_dist_df.filter(pl.col('isotype') == iso)['n_mut_H'].to_numpy()
    if len(vals) > 0:
        ax.hist(vals, bins=range(0, 60), density=True, alpha=0.55,
                color=ISO_COLORS[iso], label=f'{iso} (n={len(vals):,})', histtype='stepfilled')

ax.set_xlabel('VH codon mutations')
ax.set_ylabel('Density')
ax.set_title('VH mutation load distributions by isotype (IgM/IgG/IgA/IgE)')
ax.set_xlim(0, 55)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "fig_v4_isotype_mutation_distributions.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

## Step 1 Summary

| Calculation | Output table | Output figure |
|-------------|-------------|---------------|
| V1: S5F mutability | `mutability_profile_H.csv` | `fig_v1_mutability_profile.png` |
| V2: R_AID per germline | `evolvability_index.csv` | `fig_v2_raid_top40.png` |
| V3: CDRH3 length vs SHM | `cdrh3_length_vs_shm.csv` | `fig_v3_cdrh3_length_vs_shm.png` |
| V4: Mutation by isotype (bar) | `isotype_mutation_depth.csv` | `fig_v4_isotype_mutation_depth.png` |
| V4: Mutation by isotype (dist) | `fig_v4_isotype_mutation_distributions.csv` | `fig_v4_isotype_mutation_distributions.png` |

**Next step:** `02_phi_structure.ipynb` — Structural penalty Φ_S (S1–S3)